# 1. 파일 불러오기 
## 1. 파일 타입 변환

In [193]:
import pandas as pd
import numpy as np
import glob
import os
from hossam import *

from matplotlib import pyplot as plt
from matplotlib import font_manager as fm # 글꼴을 시스템에 등록
import seaborn as sb

In [194]:
pop = pd.read_csv(r"C:\Users\wodyd\Desktop\파이널프로젝트 자료\합본\population.csv")
cpi = pd.read_csv(r"C:\Users\wodyd\Desktop\파이널프로젝트 자료\합본\cpi.csv")
grdp = pd.read_excel(r"C:\Users\wodyd\Desktop\파이널프로젝트 자료\합본\GRDP.xlsx")
ltc = pd.read_excel(r"C:\Users\wodyd\Desktop\파이널프로젝트 자료\합본\ltc_cost.xlsx")

In [195]:
grdp = pd.read_excel(r"C:\Users\wodyd\Desktop\파이널프로젝트 자료\합본\GRDP.xlsx")

grdp.to_csv(r"C:\Users\wodyd\Desktop\파이널프로젝트 자료\합본\GRDP", index=False, encoding='utf-8-sig')

In [196]:
ltc_cost = pd.read_excel(r"C:\Users\wodyd\Desktop\파이널프로젝트 자료\합본\ltc_cost.xlsx")

ltc_cost.to_csv(r"C:\Users\wodyd\Desktop\파이널프로젝트 자료\합본\ltc_cost", index=False, encoding='utf-8-sig')

## 2. 데이터 전처리
### 1. population

In [197]:
second_col = pop.columns[1]
print("두 번째 열 이름:", second_col)

for x in pop[second_col].head(50).tolist():
    print(repr(x))

두 번째 열 이름: 연령별
'연령별'
'계'
'65세'
'66세'
'67세'
'68세'
'69세'
'70세'
'71세'
'72세'
'73세'
'74세'
'75세'
'76세'
'77세'
'78세'
'79세'
'80세'
'81세'
'82세'
'83세'
'84세'
'85세'
'86세'
'87세'
'88세'
'89세'
'90세'
'91세'
'92세'
'93세'
'94세'
'95세'
'96세'
'97세'
'98세'
'99세'
'100세 이상'
'고령화율'
'계'
'65세'
'66세'
'67세'
'68세'
'69세'
'70세'
'71세'
'72세'
'73세'
'74세'


In [198]:
second_col = pop.columns[1]

tmp = pop.copy()
tmp[second_col] = (
    tmp[second_col]
    .astype(str)
    .str.strip()
    .str.replace("\ufeff", "", regex=False)
)

print(tmp[tmp[second_col].str.contains("고령", na=False)].head(30))
print(tmp[tmp[second_col].str.contains("고령", na=False)].shape)

         시도별   연령별         2010         2011         2012         2013  \
38        전국  고령화율   10.6172141  10.91638343  11.28805028  11.78992598   
76     서울특별시  고령화율  9.213550409  9.753506189  10.25930002  10.90796487   
77     서울특별시  고령인구     10213153     10312835     10250134     10195064   
115    부산광역시  고령화율  10.82703787  11.28928155  11.84172085  12.57224384   
116    부산광역시  고령인구      3542631      3566560      3549501      3538191   
154    대구광역시  고령화율  9.768136637  10.04820326  10.42441905  11.01129965   
155    대구광역시  고령인구      2491683      2512071      2508370      2504945   
193    인천광역시  고령화율  8.351665194  8.634279833  8.984815035   9.43022566   
194    인천광역시  고령인구      2714309      2761122      2804287      2846899   
232    광주광역시  고령화율  8.683968508  8.984775199  9.372605034  9.886800616   
233    광주광역시  고령인구      1435300      1456308      1465313      1470061   
271    대전광역시  고령화율  8.418566467  8.677559318  9.001060437   9.42500367   
272    대전광역시  고령인구      1484849      1

In [199]:
# 컬럼명 간단히 바꾸기
pop.columns = ["region", "age"] + [str(y) for y in range(2010, 2025)]

# 문자열 정리
pop["region"] = pop["region"].astype(str).str.strip().str.replace("\ufeff", "", regex=False)
pop["age"] = pop["age"].astype(str).str.strip().str.replace("\ufeff", "", regex=False)

# 단위행 제거
pop = pop[pop["region"] != "시도별"].copy()

# 전국 제거
pop = pop[pop["region"] != "전국"].copy()

# 고령화율 행만 추출
pop_rate = pop[pop["age"].str.contains("고령", na=False)].copy()

# wide -> long
pop_panel = pop_rate.melt(
    id_vars=["region", "age"],
    value_vars=[str(y) for y in range(2010, 2025)],
    var_name="year",
    value_name="aging_rate"
)

# 타입 정리
pop_panel["year"] = pop_panel["year"].astype(int)
pop_panel["aging_rate"] = pd.to_numeric(pop_panel["aging_rate"], errors="coerce")

# 최종 컬럼
pop_panel = pop_panel[["region", "year", "aging_rate"]]

print(pop_panel.head(20))
print(pop_panel.shape)

     region  year   aging_rate
0     서울특별시  2010        9.214
1     서울특별시  2010 10213153.000
2     부산광역시  2010       10.827
3     부산광역시  2010  3542631.000
4     대구광역시  2010        9.768
5     대구광역시  2010  2491683.000
6     인천광역시  2010        8.352
7     인천광역시  2010  2714309.000
8     광주광역시  2010        8.684
9     광주광역시  2010  1435300.000
10    대전광역시  2010        8.419
11    대전광역시  2010  1484849.000
12    울산광역시  2010        6.599
13    울산광역시  2010  1115625.000
14  세종특별자치시  2010        0.000
15  세종특별자치시  2010        0.000
16      경기도  2010        8.446
17      경기도  2010 11484241.000
18  강원특별자치도  2010       14.413
19  강원특별자치도  2010  1512088.000
(510, 3)


In [200]:
pop_rate = pop[pop["age"].astype(str).str.strip() == "고령화율"].copy()

In [201]:
pop_panel = pop_rate.melt(
    id_vars=["region", "age"],
    value_vars=[str(y) for y in range(2010, 2025)],
    var_name="year",
    value_name="aging_rate"
)

pop_panel["year"] = pop_panel["year"].astype(int)
pop_panel["aging_rate"] = pd.to_numeric(pop_panel["aging_rate"], errors="coerce")
pop_panel = pop_panel[["region", "year", "aging_rate"]]

print(pop_panel.shape)
print(pop_panel.head(20))

(255, 3)
     region  year  aging_rate
0     서울특별시  2010       9.214
1     부산광역시  2010      10.827
2     대구광역시  2010       9.768
3     인천광역시  2010       8.352
4     광주광역시  2010       8.684
5     대전광역시  2010       8.419
6     울산광역시  2010       6.599
7   세종특별자치시  2010       0.000
8       경기도  2010       8.446
9   강원특별자치도  2010      14.413
10     충청북도  2010      13.010
11     충청남도  2010      14.808
12  전북특별자치도  2010      15.008
13     전라남도  2010      18.016
14     경상북도  2010      15.476
15     경상남도  2010      11.661
16  제주특별자치도  2010      11.902
17    서울특별시  2011       9.754
18    부산광역시  2011      11.289
19    대구광역시  2011      10.048


In [202]:
display(pop_panel)

,region,year,aging_rate
0,서울특별시,2010,9.214
1,부산광역시,2010,10.827
2,대구광역시,2010,9.768
3,인천광역시,2010,8.352
4,광주광역시,2010,8.684
...,...,...,...
250,전북특별자치도,2024,24.196
251,전라남도,2024,26.176
252,경상북도,2024,24.799
253,경상남도,2024,20.701


### 2. 소비자물가지수(cpi)

In [203]:
cpi_panel = cpi.copy()
# 컬럼 공백 제거
cpi_panel.columns = cpi_panel.columns.str.strip()

# 컬럼 이름 정리
cpi_panel = cpi_panel.rename(columns={
    cpi_panel.columns[0]: 'year',
    cpi_panel.columns[1]: 'cpi'
})

# 숫자형 변환
cpi_panel['year'] = pd.to_numeric(cpi_panel['year'], errors='coerce')
cpi_panel['cpi'] = pd.to_numeric(cpi_panel['cpi'], errors='coerce')

# 결측 제거
cpi_panel = cpi_panel.dropna()

# 인덱스 정리
cpi_panel = cpi_panel.reset_index(drop=True)

display(cpi_panel)

,year,cpi
0,2010,86.373
1,2011,89.850
2,2012,91.815
3,2013,93.010
4,2014,94.196
5,2015,94.861
6,2016,95.783
7,2017,97.645
8,2018,99.086
9,2019,99.466


### 3. 요양급여비(ltc_cost) 

In [204]:
display(ltc_cost)

,시도별,요양급여비,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
0,전국,요양급여비,2402287301.350,2588192050,2717747764.860,3082993241,3498139433,3981617681,4417672705.000,5093694004.540,6299189004.400,7736306332.880,8882675046.920,10095748015.870,11444191434.270,13192322556.380,14767543612.300
1,서울특별시,요양급여비,387148917.000,421173772,438877148.000,491413932,561236392,635821995,634179711.000,710054863.190,849761350.520,1000208031.140,1093077958.250,1202308296.800,1358025254.800,1570266152.890,1734610900.410
2,부산광역시,요양급여비,144453176.420,139715016,136461429.000,148605470,162650702,178131597,199281812.000,236408708.430,301225727.440,382941479.340,456169508.340,531990551.070,608714959.220,707112639.070,799348835.080
3,대구광역시,요양급여비,95421098.570,107057201,118860211.000,136751605,153572419,173035131,199178981.200,232985068.330,296733888.250,369797981.750,417408562.850,482284984.610,548790874.640,638301980.540,724831990.150
4,인천광역시,요양급여비,136434657.820,146496356,156926662.000,181894533,207948044,236459028,277689314.510,325396288.060,406126647.740,495934215.010,570313674.400,645753511.950,740985315.400,864583119.730,969981451.130
5,광주광역시,요양급여비,73024072.310,73976539,73352038.000,79292610,87829365,99649845,114131306.240,132646356.020,165164511.610,203472076.660,240300793.050,276114977.370,308091616.400,355159138.410,402233768.470
6,대전광역시,요양급여비,77314398.590,81774461,81977722.000,93425377,78594560,117898386,142359314.140,159457479.510,192714242.100,238207772.260,272195546.900,303396588.010,332314647.660,380813786.820,431249415.780
7,울산광역시,요양급여비,31999340.860,33675781,33705813.000,36628762,40150938,44726367,46815377.840,55698787.460,72150889.930,91604564.670,113370243.930,135942338.770,158344275.160,184320126.710,206745993.180
8,세종특별자치시,요양급여비,0.000,0,7539530.000,8339055,10227640,12808423,8234234.850,10232547.250,13886409.060,18762367.660,23981595.310,30385901.790,36047528.260,43012756.700,49027068.060
9,경기도,요양급여비,510304972.230,558480600,534841650.000,664560713,763966329,890674427,1085672378.060,1266807813.810,1573406582.880,1935884606.330,2208597362.870,2486236996.090,2858004526.520,3340874560.170,3792047955.700


In [205]:
# 원본 복사
ltc_panel = ltc_cost.copy()

# 컬럼 공백 제거
ltc_panel.columns = ltc_panel.columns.astype(str).str.strip()

# 첫 컬럼을 지역 컬럼으로 설정
region_col = ltc_panel.columns[0]

# 지역명 공백 제거
ltc_panel[region_col] = ltc_panel[region_col].astype(str).str.strip()

# wide → long 변환
ltc_panel = ltc_panel.melt(
    id_vars=region_col,
    var_name='year',
    value_name='ltc_cost'
)

# 컬럼 이름 변경
ltc_panel.rename(columns={region_col: 'region'}, inplace=True)

# 전국 제거
ltc_panel = ltc_panel[ltc_panel['region'] != '전국']

# 숫자 변환
ltc_panel['year'] = pd.to_numeric(ltc_panel['year'], errors='coerce')

ltc_panel['ltc_cost'] = (
    ltc_panel['ltc_cost']
    .astype(str)
    .str.replace(',', '', regex=False)
)

ltc_panel['ltc_cost'] = pd.to_numeric(ltc_panel['ltc_cost'], errors='coerce')

# 결측 제거
ltc_panel = ltc_panel.dropna(subset=['year','ltc_cost'])

ltc_panel['year'] = ltc_panel['year'].astype(int)

ltc_panel = ltc_panel.reset_index(drop=True)

display(ltc_panel)

,region,year,ltc_cost
0,서울특별시,2010,387148917.000
1,부산광역시,2010,144453176.420
2,대구광역시,2010,95421098.570
3,인천광역시,2010,136434657.820
4,광주광역시,2010,73024072.310
...,...,...,...
250,전북특별자치도,2024,715763531.820
251,전라남도,2024,738308968.690
252,경상북도,2024,1044443431.060
253,경상남도,2024,1031327977.800


### 4. grdp

In [206]:
display(grdp)

,시도별,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
0,전국,27996,29125,30078,31229,32320,34169,35825,37690,38922,39467,39789,42963,45045,46640,49483
1,서울특별시,32407,33740,34682,35553,36635,38911,41359,43731,46547,48248,49165,53373,56094,58829,61215
2,부산광역시,20459,20883,21761,22554,23991,25707,26800,27210,28355,29436,29272,31950,34465,35431,37085
3,대구광역시,17526,18399,19338,20260,21114,22685,23382,24170,24820,25653,25928,27770,29654,30664,31374
4,인천광역시,23871,24103,24508,25177,26559,28376,29864,31198,31688,32272,32292,35388,38065,39104,41187
5,광주광역시,19867,20426,21203,22248,23448,24731,26248,27449,28594,29807,30823,32098,33497,35143,37675
6,대전광역시,20428,21195,21844,22326,23047,24568,26265,27281,28027,29870,31276,32928,34409,36174,38220
7,울산광역시,58224,64781,65248,65650,63793,65408,65031,66786,65897,65486,59529,70889,76234,80821,85194
8,세종특별자치시,0,0,0,59432,67352,53194,46767,44964,41137,40597,40212,42797,42207,43227,44610
9,경기도,25344,26130,27475,29030,30249,32590,34399,37146,38541,38135,38890,41392,42903,43155,46996


In [207]:
grdp.columns = [str(c).strip() for c in grdp.columns]

region_col = grdp.columns[0]

grdp[region_col] = grdp[region_col].astype(str).str.strip()

grdp_panel = grdp.melt(
    id_vars=region_col,
    var_name='year',
    value_name='grdp'
)

grdp_panel.rename(columns={region_col: 'region'}, inplace=True)

grdp_panel['region'] = grdp_panel['region'].astype(str).str.strip()
grdp_panel = grdp_panel[grdp_panel['region'] != '전국']

grdp_panel['year'] = pd.to_numeric(grdp_panel['year'], errors='coerce')
grdp_panel['grdp'] = (
    grdp_panel['grdp']
    .astype(str)
    .str.replace(',', '', regex=False)
    .str.strip()
)
grdp_panel['grdp'] = pd.to_numeric(grdp_panel['grdp'], errors='coerce')

grdp_panel = grdp_panel.dropna(subset=['year', 'grdp'])
grdp_panel['year'] = grdp_panel['year'].astype(int)

grdp_panel.head()

,region,year,grdp
1,서울특별시,2010,32407
2,부산광역시,2010,20459
3,대구광역시,2010,17526
4,인천광역시,2010,23871
5,광주광역시,2010,19867


## 3. 패널 데이터
### 1. 데이터 합치기

In [208]:
final_panel = pop_panel.merge(
    grdp_panel,
    on=['region', 'year'],
    how='left'
)

final_panel = final_panel.merge(
    ltc_panel,
    on=['region', 'year'],
    how='left'
)

final_panel = final_panel.merge(
    cpi_panel,
    on='year',
    how='left'
)
display(final_panel)

,region,year,aging_rate,grdp,ltc_cost,cpi
0,서울특별시,2010,9.214,32407,387148917.000,86.373
1,부산광역시,2010,10.827,20459,144453176.420,86.373
2,대구광역시,2010,9.768,17526,95421098.570,86.373
3,인천광역시,2010,8.352,23871,136434657.820,86.373
4,광주광역시,2010,8.684,19867,73024072.310,86.373
...,...,...,...,...,...,...
250,전북특별자치도,2024,24.196,37980,715763531.820,114.180
251,전라남도,2024,26.176,59177,738308968.690,114.180
252,경상북도,2024,24.799,52298,1044443431.060,114.180
253,경상남도,2024,20.701,46550,1031327977.800,114.180


### 2. post_2017(정책변수) 추가

In [209]:
final_panel['post_2017'] = (final_panel['year'] >= 2017).astype(int)
display(final_panel)

,region,year,aging_rate,grdp,ltc_cost,cpi,post_2017
0,서울특별시,2010,9.214,32407,387148917.000,86.373,0
1,부산광역시,2010,10.827,20459,144453176.420,86.373,0
2,대구광역시,2010,9.768,17526,95421098.570,86.373,0
3,인천광역시,2010,8.352,23871,136434657.820,86.373,0
4,광주광역시,2010,8.684,19867,73024072.310,86.373,0
...,...,...,...,...,...,...,...
250,전북특별자치도,2024,24.196,37980,715763531.820,114.180,1
251,전라남도,2024,26.176,59177,738308968.690,114.180,1
252,경상북도,2024,24.799,52298,1044443431.060,114.180,1
253,경상남도,2024,20.701,46550,1031327977.800,114.180,1


### 3. 실질요양급여(ltc_real) 추가
- 물가 영향을 제거한 장기요양보험 지출
- ltc_cost와 ltc_real의 값은 다르다
    - 2024년 값은 유지
    - 2010~2020은 ltc_real > ltc_cost

In [211]:
base_cpi = cpi_panel.loc[cpi_panel['year'] == 2024, 'cpi'].iloc[0]

final_panel['ltc_real'] = (
    final_panel['ltc_cost'] * base_cpi / final_panel['cpi']
)
display(final_panel)

,region,year,aging_rate,grdp,ltc_cost,cpi,post_2017,ltc_real
0,서울특별시,2010,9.214,32407,387148917.000,86.373,0,511787981.696
1,부산광역시,2010,10.827,20459,144453176.420,86.373,0,190958559.777
2,대구광역시,2010,9.768,17526,95421098.570,86.373,0,126141051.425
3,인천광역시,2010,8.352,23871,136434657.820,86.373,0,180358552.208
4,광주광역시,2010,8.684,19867,73024072.310,86.373,0,96533506.725
...,...,...,...,...,...,...,...,...
250,전북특별자치도,2024,24.196,37980,715763531.820,114.180,1,715763531.820
251,전라남도,2024,26.176,59177,738308968.690,114.180,1,738308968.690
252,경상북도,2024,24.799,52298,1044443431.060,114.180,1,1044443431.060
253,경상남도,2024,20.701,46550,1031327977.800,114.180,1,1031327977.800


### 4. 2010, 2011 세종특별자치도 데이터 제거
- 패널 모델은 대부분 unbalanced panel을 허용
    - 금지    
        - 0으로 채우기
        - 값 대체 하기
    

In [212]:
final_panel['ltc_real'] = final_panel['ltc_real'].replace(0, np.nan)

final_panel['log_ltc'] = np.log(final_panel['ltc_real'])

final_panel = final_panel.replace([np.inf, -np.inf], np.nan)

final_panel = final_panel.dropna(subset=['log_ltc'])

display(final_panel)

,region,year,aging_rate,grdp,ltc_cost,cpi,post_2017,ltc_real,log_ltc
0,서울특별시,2010,9.214,32407,387148917.000,86.373,0,511787981.696,20.053
1,부산광역시,2010,10.827,20459,144453176.420,86.373,0,190958559.777,19.068
2,대구광역시,2010,9.768,17526,95421098.570,86.373,0,126141051.425,18.653
3,인천광역시,2010,8.352,23871,136434657.820,86.373,0,180358552.208,19.010
4,광주광역시,2010,8.684,19867,73024072.310,86.373,0,96533506.725,18.385
...,...,...,...,...,...,...,...,...,...
250,전북특별자치도,2024,24.196,37980,715763531.820,114.180,1,715763531.820,20.389
251,전라남도,2024,26.176,59177,738308968.690,114.180,1,738308968.690,20.420
252,경상북도,2024,24.799,52298,1044443431.060,114.180,1,1044443431.060,20.767
253,경상남도,2024,20.701,46550,1031327977.800,114.180,1,1031327977.800,20.754


## 3. 기초통계량
### 1. 데이터 정보 파악

In [213]:
print(f"데이터셋 크기: {final_panel.shape}")
print(f"열 개수: {final_panel.shape[1]}")
print(f"행 개수: {final_panel.shape[0]}")
print(final_panel.info())


데이터셋 크기: (253, 9)
열 개수: 9
행 개수: 253
<class 'pandas.core.frame.DataFrame'>
Index: 253 entries, 0 to 254
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   region      253 non-null    object 
 1   year        253 non-null    int32  
 2   aging_rate  253 non-null    float64
 3   grdp        253 non-null    int64  
 4   ltc_cost    253 non-null    float64
 5   cpi         253 non-null    float64
 6   post_2017   253 non-null    int32  
 7   ltc_real    253 non-null    float64
 8   log_ltc     253 non-null    float64
dtypes: float64(5), int32(2), int64(1), object(1)
memory usage: 17.8+ KB
None


### 2. 결측치 확인

In [214]:
from pandas import DataFrame

final_panel.isna().sum()

region        0
year          0
aging_rate    0
grdp          0
ltc_cost      0
cpi           0
post_2017     0
ltc_real      0
log_ltc       0
dtype: int64

### 3. 수치형 변수 기술 통계 확인

In [215]:
final_panel[['aging_rate', 'grdp', 'ltc_cost','ltc_real','cpi']].describe().T

,count,mean,std,min,25%,50%,75%,max
aging_rate,253.000,14.816,4.277,0.000,11.629,14.808,17.570,26.176
grdp,253.000,37509.941,13087.004,0.000,27817.000,34682.000,43582.000,85194.000
ltc_cost,253.000,395745551.857,488224261.126,7539530.000,133531335.660,236359639.890,518158772.470,3792047955.700
ltc_real,253.000,441558572.649,515451565.007,9376066.388,163258151.855,274888959.652,577203596.494,3792047955.700
cpi,253.000,98.621,7.550,86.373,93.010,97.645,102.500,114.180


In [216]:
result = hs_stats.describe(final_panel,'aging_rate', 'grdp', 'ltc_cost','ltc_real','cpi')
print(result)

             count  na_count  na_rate          mean           std         min  \
aging_rate 253.000         0    0.000        14.816         4.277       0.000   
grdp       253.000         0    0.000     37509.941     13087.004       0.000   
ltc_cost   253.000         0    0.000 395745551.857 488224261.126 7539530.000   
ltc_real   253.000         0    0.000 441558572.649 515451565.007 9376066.388   
cpi        253.000         0    0.000        98.621         7.550      86.373   

                     25%           50%           75%            max  \
aging_rate        11.629        14.808        17.570         26.176   
grdp           27817.000     34682.000     43582.000      85194.000   
ltc_cost   133531335.660 236359639.890 518158772.470 3792047955.700   
ltc_real   163258151.855 274888959.652 577203596.494 3792047955.700   
cpi               93.010        97.645       102.500        114.180   

                     iqr             up           down  outlier_count  \
aging_rate   

#### 인사이트
- 데이터 완성도: 결측치 0개 
- 데이터 분포 특성 
    - aging_rate: 로그변환 불필요
    - ltc_cost, ltc_real: 로그변환 필요
    - grdp, cpi: 로그변환 불필요(grdp의 경우 둘 다 많이 쓰긴 하지만, 패널 데이터에서는 그냥 사용)

### 4. 로그 변환

In [217]:
import numpy as np
final_panel['log_ltc'] = np.log(final_panel['ltc_real'])
display(final_panel)

,region,year,aging_rate,grdp,ltc_cost,cpi,post_2017,ltc_real,log_ltc
0,서울특별시,2010,9.214,32407,387148917.000,86.373,0,511787981.696,20.053
1,부산광역시,2010,10.827,20459,144453176.420,86.373,0,190958559.777,19.068
2,대구광역시,2010,9.768,17526,95421098.570,86.373,0,126141051.425,18.653
3,인천광역시,2010,8.352,23871,136434657.820,86.373,0,180358552.208,19.010
4,광주광역시,2010,8.684,19867,73024072.310,86.373,0,96533506.725,18.385
...,...,...,...,...,...,...,...,...,...
250,전북특별자치도,2024,24.196,37980,715763531.820,114.180,1,715763531.820,20.389
251,전라남도,2024,26.176,59177,738308968.690,114.180,1,738308968.690,20.420
252,경상북도,2024,24.799,52298,1044443431.060,114.180,1,1044443431.060,20.767
253,경상남도,2024,20.701,46550,1031327977.800,114.180,1,1031327977.800,20.754


## 3. 상관분석 수행
### 1. 상관분석

In [218]:
final_panel[['aging_rate','grdp','log_ltc']].corr()

,aging_rate,grdp,log_ltc
aging_rate,1.000,0.279,0.502
grdp,0.279,1.000,0.072
log_ltc,0.502,0.072,1.000


### 2. 다중공선성 확인(VIF)

In [219]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm
x = final_panel[['aging_rate','grdp','post_2017']]
x = sm.add_constant(x)

vif_dict = {
    col: variance_inflation_factor(x.values, i)
    for i, col in enumerate(x.columns)
}

print(vif_dict)

{'const': 18.51950041860227, 'aging_rate': 1.3192438375708948, 'grdp': 1.169192533263173, 'post_2017': 1.3980651222598401}


#### 인사이트
- 독립변수끼리 비교하였을 때, 다중공선성(VIF)값은 < 5 이므로 진행 ㄱㄱ
- post_2017은 정책 더미 변수이며 동시에 통제변수로 사용

## 4. Pooled OLS(기본 회귀)
### 1. 기본 관계 분석
- 패널 데이터를 일반 회귀처럼 취급하는 모델
- region x year 구조를 무시하고 전체 관측치로 보는 모델
- 패널 분석 진행 순서: Pooled OLS(패널 구조 무시) -> Fixed Effects(지역 고정효과 통제) -> Random Effects(랜덤 효과)
    - 패널 효과가 실제로 존재하는가에 대해 확인하기 위해

- Fixed Effects(지역 고정효과): 서울은 원래 노인이 많아 의료 이용이 높을 수 있다. 이게 '서울 특성'아며 LTC 증가의 원인이 될 수 있다. 그래서 OLS -> Fixed Effects 해서 비교

In [220]:
import statsmodels. formula.api as smf
ols = smf.ols(
    'log_ltc ~ aging_rate + grdp + post_2017',
    data=final_panel
).fit()

print(ols.summary())

                            OLS Regression Results                            
Dep. Variable:                log_ltc   R-squared:                       0.310
Model:                            OLS   Adj. R-squared:                  0.301
Method:                 Least Squares   F-statistic:                     37.21
Date:                    화, 10 3 2026   Prob (F-statistic):           6.67e-20
Time:                        19:24:07   Log-Likelihood:                -326.10
No. Observations:                 253   AIC:                             660.2
Df Residuals:                     249   BIC:                             674.3
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     18.0222      0.239     75.258      0.0

#### 인사이트
- 모델 유의성
- F-statistic: 37.21
- Prob(F-statistic): 6.67e-20
   - --> 설명변수들이 종속변수(log_ltc)에 영향을 주지 않는다. 즉, 모델 전체는 유의하다

--------------------------
- 설명력 (R^2)
- R-squared: 0.310
- Adj, R-squared: 0.301
   - --> 이 모델은 장기요양보험 지출 변동의 약 31%를 설명한다
--------------------------
- 핵심 변수 해석
- coef = 0.1019
- p-value = 0.000
   - --> 고령화율이 1%p 증가하면 장기요양보험 지출은 약 10% 증가한다
--------------------------
- grdp
- coef = -1.133e-05
- p = 0.015
   - --> 유의(5% 수준)
   - --> 그러나 계수가 매우 작다.
   - ---> GRDP 증가가 LTC 지출과 약한 음의 관계를 가진다
        - 해석
            - 경제력 높은 지역 --> 건강 상태 좋음
            - LTC 이용 감소 가능성
--------------------------
- post_2017
- coef = 0.5737(로그 모델이므로 풀어주면 1.77)
- p = 0.000
   - --> 2017년 정책 이후 LTC 지출 약 77% 증가
   - --> 장기요양 등급 개편의 가능성이 있음
--------------------------
- 잔차 정규성
- Prob(JB): 9.08e-08
   - --> 정규성은 약간 깨져 있으나 표본 수(253)이므로 중심극한정리에 의해 큰 문제는 아니다
--------------------------
- Durbin-Watson
- 2.814
   - --> > 2 이므로 약한 음의 자기성관(큰 문제는 아님)    